In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("dynamic_pricing_data.csv")

#linear regression

In [ ]:
X = df.drop("Price", axis=1)
y = df["Price"]

In [ ]:
X = pd.get_dummies(
    X,
    columns=["Season", "DayTime"],
    drop_first=True
)

In [ ]:
print(X.columns)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.select_dtypes(include=["object", "string"]).columns)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred =  model.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
r2Score = r2_score(y_test,y_pred)
print("r2_score:",r2Score)
mae = mean_absolute_error(y_test,y_pred)
print("mean absolute error:",mae)
mse = mean_squared_error(y_test, y_pred)
print("MSE:", mse)
rmse = np.sqrt(mse)
print("RMSE:", rmse)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

residuals = y_test - y_pred

plt.figure(figsize=(7,5))
sns.histplot(residuals, kde=True)

plt.title("Distribution of Residuals")
plt.xlabel("Residual")
plt.show()

Model Performance

The Multiple Linear Regression model performed exceptionally well on the test dataset. It achieved an R² score of 0.9884, indicating that approximately 98.84% of the variation in ride prices is explained by the selected input features.

The model achieved a Mean Absolute Error (MAE) of 10.81, meaning that, on average, the predicted ride price differs from the actual price by approximately ₹11.

The Root Mean Squared Error (RMSE) was 13.90, indicating that the model maintains a low prediction error across the dataset.

Since the dataset was synthetically generated with realistic relationships between features such as demand, stock, distance, discounts, and competitor pricing, the high predictive performance is expected.

#decision tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train,y_train)

In [ ]:
y_pred = dt_model.predict(X_test)

In [ ]:
r2Score = r2_score(y_test,y_pred)
print("r2_score:",r2Score)
mae = mean_absolute_error(y_test,y_pred)
print("mean absolute error:",mae)
mse = mean_squared_error(y_test, y_pred)
print("MSE:", mse)
rmse = np.sqrt(mse)
print("RMSE:", rmse)

In [ ]:
train_pred = dt_model.predict(X_train)
test_pred = dt_model.predict(X_test)

print("Train R²:", r2_score(y_train, train_pred))
print("Test R² :", r2_score(y_test, test_pred))

The model achieved a perfect training R² score of 1.00, meaning it predicted every training sample correctly.
However, achieving perfect accuracy on the training data is generally not desirable, because it indicates that the model has memorized the training dataset instead of learning general patterns.
This behavior is known as overfitting.
Although the test R² score remained high (0.9858), the gap between the training and testing performance suggests that the tree learned many dataset-specific splits.

#random forest

In [ ]:
from sklearn.ensemble import  RandomForestRegressor
rf_model = RandomForestRegressor()
rf_model.fit(X_train,y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
r2Score = r2_score(y_test,y_pred_rf)
print("r2_score:",r2Score)
mae = mean_absolute_error(y_test,y_pred_rf)
print("mean absolute error:",mae)
mse = mean_squared_error(y_test, y_pred_rf)
print("MSE:", mse)
rmse = np.sqrt(mse)
print("RMSE:", rmse)

In [ ]:
train_pred = rf_model.predict(X_train)

print("Train R²:", r2_score(y_train, train_pred))
print("Test R² :", r2_score(y_test, y_pred_rf))

In [ ]:
from sklearn.model_selection import cross_val_score

# Compare train vs test performance
train_r2 = rf_model.score(X_train, y_train)
test_r2 = rf_model.score(X_test, y_test)
print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")

# 5-fold cross-validation for a more robust estimate
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='r2')
print(f"CV R² scores: {cv_scores}")
print(f"CV R² mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
import joblib
joblib.dump(rf_model, 'model.pkl')

The Random Forest Regressor achieved the highest predictive performance among all the models. Although the training R² score is very high (0.9992), the test R² score (0.9941) remains very close, resulting in a small train-test gap of approximately 0.005. This indicates that the model generalizes well and shows minimal overfitting

## Model Comparison

| Model | R² | MAE | MSE | RMSE |
|---|---|---|---|---|
| Linear Regression | 0.9884 | 10.81 | 193.17 | 13.90 |
| Decision Tree (unpruned) | 0.9858 | 11.59 | 235.79 | 15.36 |
| Random Forest | 0.9941 | 7.70 | 97.33 | 9.87 |

Random Forest was selected as the final model. It outperforms both alternatives on every 
metric, and unlike the unpruned Decision Tree, it does so while maintaining a small 
train-test gap (0.0051), indicating this performance is genuine and generalizes to unseen 
data rather than being a product of overfitting.

Notably, the unpruned Decision Tree — despite achieving a perfect training R² of 1.0000 — 
performed *worse* on the test set than the simpler Linear Regression model. This highlights 
that training accuracy alone is not a reliable indicator of real-world performance; a model 
that memorizes its training data can generalize worse than a simpler model that doesn't.

**Final model: Random Forest Regressor** (R²=0.9941, MAE=₹7.70)

In [ ]:
importances = rf_model.feature_importances_
feature_names = X_train.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print(importance_df)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Importance")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.show()